# Building and querying a knowledge graph

**Time**: ~45-60 minutes, most of it waiting on model calls. **Cost**: a few cents at most on Gemini's cheapest models -- this notebook makes on the order of 30-40 small calls total across extraction, entity resolution, querying, and the comparison section near the end. If you haven't run [`setup_guide.ipynb`](setup_guide.ipynb) yet, do that first: this notebook assumes a working `driver` (graph database connection) and `call_llm()` function.

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../session_1/setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

This week's overview described the shape of the problem: a question that crosses two departments usually has an answer that already exists, scattered across documents that were never written to talk to each other. This notebook builds that scenario in miniature, from a fictional outdoor-gear company, **Fernwood Outfitters**, and then builds and queries a knowledge graph from it.

**The documents** (10 total, all invented): two product pages, a returns-and-shipping policy, a customer-segments reference, two sets of meeting notes, and four support tickets. No single document answers the question a support lead actually needs answered: *which customers who complained about the TrailRunner backpack during the Summer Trail Sale are entitled to what, under which policy clause, and did the campaign's own messaging make things worse?* The product page doesn't know about tickets. The policy doesn't know which customers it applies to. The meeting notes don't know the shipping SLA was breached until a person reads a ticket and a policy side by side. That's the gap a knowledge graph closes -- not by adding new information, but by making the relationships between documents into something a query can walk across.

**What you'll do:**
1. Use an LLM to extract entities and relationships from each document.
2. Merge the duplicate entities that extraction inevitably produces (the same customer or product named slightly differently across documents).
3. Load the result into your graph database and query it in Cypher, including a query that needs more than one hop to answer.
4. Compare that against what a flat table would have made easy or hard.
5. Use the graph as the source a model draws facts from to answer a question -- GraphRAG -- and check whether the answer is actually supported by what's in the graph.
6. Measure that against two alternatives on the same question -- no retrieval at all, and standard vector similarity RAG -- with the correct answer known in advance, to see exactly where and why each one goes wrong.

## Reconnect

If you're continuing in a fresh Colab runtime, rerun this cell to restore `driver` and `call_llm()` from the setup notebook. If you're still in the same runtime you ran `setup_guide.ipynb` in, this just confirms both still work.

In [ ]:
from neo4j import GraphDatabase
from google.colab import userdata

driver = GraphDatabase.driver(
    userdata.get("NEO4J_URI"),
    auth=(userdata.get("NEO4J_USERNAME"), userdata.get("NEO4J_PASSWORD")),
)

LLM_PROVIDER = "gemini"  # change this one value to swap providers -- see setup_guide.ipynb Part 4


def call_llm(prompt: str, model: str = None) -> str:
    if LLM_PROVIDER == "gemini":
        from google import genai

        client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
        response = client.models.generate_content(
            model=model or "gemini-2.5-flash-lite", contents=prompt
        )
        return response.text
    elif LLM_PROVIDER == "openai":
        from openai import OpenAI

        client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
        response = client.chat.completions.create(
            model=model or "gpt-4.1-mini", messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    elif LLM_PROVIDER == "anthropic":
        import anthropic

        client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
        response = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text
    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}")


with driver.session() as session:
    session.run("RETURN 1").single()
print("Graph database and LLM connection both ready. LLM_PROVIDER =", LLM_PROVIDER)

## The internal documents

Ten short documents, four types, one fictional company. Read a couple before moving on -- notice that "TrailRunner 32L Backpack" (product page), "TrailRunner backpack" (ticket #1077), and "TrailRunner 32L Backpack" again (ticket #1042) are the same product named three slightly different ways, and that "Maria Ibarra" (ticket #1042) and "M. Ibarra" (ticket #1103) are the same customer. Real internal documentation does this constantly -- nobody writes a support ticket with the product's exact SKU-page name in mind -- and it's the reason the "merging duplicates" step later in this notebook isn't optional busywork.

In [ ]:
DOCS = [
    {
        "doc_id": 'product_trailrunner_backpack',
        "doc_type": 'product_page',
        "title": 'TrailRunner 32L Backpack product page',
        "text": "# Product page: TrailRunner 32L Backpack\n\n**SKU**: FW-BP-3200\n**Category**: Backpacks\n**Price**: $129\n**Launched**: March 2026\n\nThe TrailRunner 32L Backpack is Fernwood Outfitters' flagship day-hiking pack, built around a 32-liter main compartment, a ventilated back panel, and a front zipper access point for quick gear changes mid-trail.\n\n**Materials**: 420D recycled nylon shell, YKK zippers.\n**Weight**: 1.4 kg.\n**Colors**: Moss Green, Slate Grey, Rust.\n\nManufactured by our contract partner Highline Textiles (Ho Chi Minh City) since March 2026. Warranty: 2 years against manufacturing defects, per Fernwood's standard returns and shipping policy.",
    },
    {
        "doc_id": 'product_alpine_rain_shell',
        "doc_type": 'product_page',
        "title": 'Alpine Rain Shell product page',
        "text": "# Product page: Alpine Rain Shell\n\n**SKU**: FW-JK-1150\n**Category**: Outerwear\n**Price**: $189\n**Launched**: January 2026\n\nA 3-layer waterproof shell built for sustained rain on exposed trail, not just a shower on the way to the car. Fully seam-taped, pit zips for ventilation, helmet-compatible hood.\n\n**Materials**: 2.5-layer proprietary membrane, DWR coating.\n**Weight**: 340 g.\n**Colors**: Storm Blue, Black.\n\nManufactured by our contract partner Highline Textiles (Ho Chi Minh City) since January 2026. Warranty: 2 years against manufacturing defects, per Fernwood's standard returns and shipping policy.",
    },
    {
        "doc_id": 'policy_returns_and_shipping',
        "doc_type": 'policy',
        "title": 'Returns and shipping policy',
        "text": '# Fernwood Outfitters: returns and shipping policy\n\n**Effective**: February 2026. **Owner**: Customer Operations.\n\n## Shipping SLA\n\nStandard shipping is quoted at 5-7 business days from order confirmation. If a shipment arrives more than 10 business days after the quoted window, the order qualifies for a **late-shipment remedy**: the customer may choose either a 20% refund on that order or a free return with full refund. This remedy applies regardless of the reason for the delay (carrier, warehouse, or demand surge).\n\n## Manufacturing defects\n\nAny item with a manufacturing defect (a fault in materials or construction, not damage from use) reported within 60 days of delivery qualifies for a **full refund and a free return**, independent of the standard 30-day return window below. Zipper failure, seam failure, and membrane delamination are treated as manufacturing defects unless there is clear evidence of misuse.\n\n## Standard returns\n\nUnworn items may be returned within 30 days of delivery for a full refund. Items showing normal wear from trail use are not eligible under this clause (see manufacturing defects above for faults distinct from wear).\n\n## Promotional orders\n\nOrders placed during a promotional campaign follow the same shipping SLA and defect policy as standard orders. A discount applied at checkout does not reduce or waive the remedies above.',
    },
    {
        "doc_id": 'segments_reference',
        "doc_type": 'reference',
        "title": 'Customer segments reference',
        "text": '# Customer segments reference\n\nMaintained by Marketing Analytics. Used for campaign targeting and email lists.\n\n- **Weekend Hikers**: customers with 1-3 orders in the past year, average order value under $200, primarily browsing day-hiking gear (backpacks, footwear, light layers).\n- **Trail Runners**: customers who have purchased running-specific or ultralight gear (rain shells, hydration vests) and engage with trail-running content.\n- **Gear Collectors**: customers with 4+ orders in the past year across multiple categories, high average order value, early adopters of new product launches.',
    },
    {
        "doc_id": 'meeting_notes_campaign_planning',
        "doc_type": 'meeting_notes',
        "title": 'Summer Trail Sale campaign planning notes',
        "text": '# Meeting notes: Summer Trail Sale campaign planning\n\n**Date**: June 2, 2026. **Attendees**: R. Okafor (Marketing), T. Novak (Marketing), S. Haddad (Merchandising).\n\nDecided to run the **Summer Trail Sale**, June 15 - July 15, 2026, at 20% off, targeting the **Weekend Hikers** segment by email and paid social. Featured products: **TrailRunner 32L Backpack** and **Alpine Rain Shell**, chosen because both had launched earlier in the year and Merchandising wanted to move summer inventory ahead of the fall line.\n\nS. Haddad flagged that Highline Textiles\' production run for the TrailRunner backpack had scaled up quickly to meet the sale\'s projected volume, and asked Ops to confirm the warehouse could handle the order surge without slipping the shipping SLA. R. Okafor noted the campaign messaging would lead with "fast, free shipping on every order" — action item for T. Novak to confirm this claim with Customer Operations before the campaign goes live.\n\nNext check-in: July 15.',
    },
    {
        "doc_id": 'meeting_notes_support_escalation',
        "doc_type": 'meeting_notes',
        "title": 'Support escalation review notes',
        "text": '# Meeting notes: support escalation review\n\n**Date**: July 15, 2026. **Attendees**: T. Novak (Marketing), P. Alvarez (Customer Operations), S. Haddad (Merchandising).\n\nP. Alvarez raised that ticket volume for the TrailRunner 32L Backpack has climbed since the Summer Trail Sale launched, with two distinct problems mixed together: a cluster of zipper failures (a manufacturing defect, per policy, not a shipping issue) and a separate cluster of late deliveries traced to the warehouse being under-staffed for the sale\'s order volume (a shipping SLA issue, different remedy).\n\nT. Novak confirmed the campaign\'s "fast, free shipping" messaging should have been checked against Customer Operations before launch, per the June 2 action item, and it was not. Agreed to pause that specific messaging line in future sends. S. Haddad will follow up with Highline Textiles on the zipper defect rate for the current production run.\n\nAction items: P. Alvarez to make sure every affected customer\'s ticket is resolved under the correct policy clause (defect vs. shipping delay); T. Novak to audit campaign copy before the next sale.',
    },
    {
        "doc_id": 'support_ticket_1042',
        "doc_type": 'support_ticket',
        "title": 'Ticket #1042 -- Maria Ibarra',
        "text": "# Support ticket #1042\n\n**Opened**: June 22, 2026. **Customer**: Maria Ibarra. **Segment**: Weekend Hikers.\n\nMaria wrote in about her TrailRunner 32L Backpack, ordered June 16 during the Summer Trail Sale. The main zipper stopped tracking after about two weeks of normal use — not caught on anything, no visible damage to the fabric around it, just stopped closing. She's asking whether this is covered and what her options are.\n\n**Status**: open, awaiting resolution under the correct policy clause.",
    },
    {
        "doc_id": 'support_ticket_1077',
        "doc_type": 'support_ticket',
        "title": 'Ticket #1077 -- David Chen',
        "text": '# Support ticket #1077\n\n**Opened**: July 3, 2026. **Customer**: David Chen. **Segment**: Weekend Hikers.\n\nDavid ordered a TrailRunner backpack on June 18 during the sale, quoted 5-7 business days. It arrived July 3 — 14 business days after confirmation. He referenced the campaign email\'s "fast, free shipping" line and wants to know what Fernwood is going to do about it.\n\n**Status**: open, awaiting resolution under the correct policy clause.',
    },
    {
        "doc_id": 'support_ticket_1090',
        "doc_type": 'support_ticket',
        "title": 'Ticket #1090 -- Priya Natarajan',
        "text": "# Support ticket #1090\n\n**Opened**: July 8, 2026. **Customer**: Priya Natarajan. **Segment**: Trail Runners.\n\nPriya ordered an Alpine Rain Shell on June 25 at full price — she's in the Trail Runners segment, which the Summer Trail Sale did not target. Delivery was quoted 5-7 business days and arrived July 8 — 12 business days later. She wants to know if she's entitled to anything given it wasn't a sale order.\n\n**Status**: open, awaiting resolution under the correct policy clause.",
    },
    {
        "doc_id": 'support_ticket_1103',
        "doc_type": 'support_ticket',
        "title": 'Ticket #1103 -- M. Ibarra follow-up',
        "text": '# Support ticket #1103\n\n**Opened**: July 20, 2026. **Customer**: M. Ibarra. **Segment**: Weekend Hikers.\n\nFollow-up from M. Ibarra (previously reported a zipper issue on her TrailRunner 32L Backpack, ticket #1042) asking whether the replacement pack under warranty will ship in the same color, Moss Green.\n\n**Status**: open, informational.',
    },
]

In [ ]:
for d in DOCS:
    print(f"[{d['doc_type']:<14}] {d['doc_id']:<32} {d['title']}")
print(f"\n{len(DOCS)} documents total.")

## Extracting entities and relationships

A **knowledge graph** stores **nodes** (entities -- a customer, a product, a policy clause) and **edges** (typed, directed relationships between them -- a customer *files* a ticket, a ticket is *about* a product). Turning a paragraph of prose into that structure is exactly the kind of task an LLM is suited for: read the text, decide what the entities are, decide how they relate, and return it as structured data instead of another paragraph of prose.

Left completely unconstrained, this drifts -- one document produces a relationship type called `RELATES_TO`, another produces `IS_CONNECTED_WITH`, and you end up with a graph that technically has edges but no two of them are reliably the same kind of edge. The fix is the same one a database designer would reach for: define the schema first, and instruct the model to extract into it rather than invent its own.

**Node types**: `Customer`, `Product`, `Ticket`, `Campaign`, `Segment`, `IssueType`, `PolicyClause`, `Company`

**Relationship types**: `FILED` (Customer→Ticket), `ABOUT_PRODUCT` (Ticket→Product), `HAS_ISSUE_TYPE` (Ticket→IssueType), `GOVERNS` (PolicyClause→IssueType), `IN_SEGMENT` (Customer→Segment), `TARGETS_SEGMENT` (Campaign→Segment), `FEATURES_PRODUCT` (Campaign→Product), `DURING_CAMPAIGN` (Ticket→Campaign), `MANUFACTURED_BY` (Product→Company)

This is a real design choice with a real tradeoff. A fixed schema keeps the graph queryable and consistent, but it means the model can only extract what the schema anticipated -- a relationship type nobody thought to define gets forced into the nearest existing one, or dropped. Production knowledge-graph pipelines usually start narrower than they'd like and widen the schema deliberately as new document types show up, rather than letting the model freelance from the start.

In [ ]:
import json

EXTRACTION_PROMPT = """You are extracting a knowledge graph from one internal company document.

Extract only entities and relationships that fit this schema. Do not invent types outside it.

Node types: Customer, Product, Ticket, Campaign, Segment, IssueType, PolicyClause, Company
Relationship types (source_type -RELATION-> target_type):
  Customer -FILED-> Ticket
  Ticket -ABOUT_PRODUCT-> Product
  Ticket -HAS_ISSUE_TYPE-> IssueType
  PolicyClause -GOVERNS-> IssueType
  Customer -IN_SEGMENT-> Segment
  Campaign -TARGETS_SEGMENT-> Segment
  Campaign -FEATURES_PRODUCT-> Product
  Ticket -DURING_CAMPAIGN-> Campaign
  Product -MANUFACTURED_BY-> Company

Use the exact names as they appear in the text (don't normalize spelling here -- a later
step handles that). IssueType values should be short category labels, e.g. "Manufacturing
defect" or "Shipping delay", not a restatement of the whole complaint. PolicyClause names
should be short labels too, e.g. "Manufacturing defects" or "Shipping SLA".

Return ONLY valid JSON, no other text, in this exact shape:
{{
  "entities": [{{"type": "Customer", "name": "Maria Ibarra"}}, ...],
  "relationships": [{{"source": "Maria Ibarra", "relation": "FILED", "target": "Ticket #1042"}}, ...]
}}

Document ({doc_type}, id={doc_id}):
{text}
"""


def extract_graph_elements(doc: dict) -> dict:
    """Call the LLM once on one document, return {"entities": [...], "relationships": [...]}."""
    prompt = EXTRACTION_PROMPT.format(doc_type=doc["doc_type"], doc_id=doc["doc_id"], text=doc["text"])
    raw = call_llm(prompt).strip()

    # Models sometimes wrap JSON in a ```json fence even when told not to -- strip it if present.
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw[raw.find("{"):]

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"Could not parse JSON from {doc['doc_id']}: {e}")
        print("Raw response:", raw[:300])
        return {"entities": [], "relationships": []}

    for e in parsed.get("entities", []):
        e["source_doc"] = doc["doc_id"]
    for r in parsed.get("relationships", []):
        r["source_doc"] = doc["doc_id"]
    return parsed

In [ ]:
all_entities = []
all_relationships = []

for doc in DOCS:
    result = extract_graph_elements(doc)
    all_entities.extend(result["entities"])
    all_relationships.extend(result["relationships"])
    print(f"{doc['doc_id']:<32} -> {len(result['entities'])} entities, {len(result['relationships'])} relationships")

print(f"\nTotal before merging: {len(all_entities)} raw entities, {len(all_relationships)} raw relationships")

In [ ]:
# Look for the duplicate-naming problem the intro flagged -- same entity, different strings.
backpack_variants = sorted({e["name"] for e in all_entities if "trailrunner" in e["name"].lower()})
ibarra_variants = sorted({e["name"] for e in all_entities if "ibarra" in e["name"].lower()})

print("Name variants extracted for the same product:", backpack_variants)
print("Name variants extracted for the same customer:", ibarra_variants)

Multiple raw strings, same real-world entity. Left as-is, a query for "everyone who filed a ticket about the TrailRunner 32L Backpack" would miss ticket #1077, because the graph would hold `TrailRunner backpack` as a node with no edge connecting it to `TrailRunner 32L Backpack`. This is the "merging the duplicates it produces" step the overview named, and it's not optional -- an unmerged graph looks complete while missing exactly the connections a real question would need.

## Merging duplicates

Two passes, cheapest first:

1. **Normalize.** Lowercase, strip whitespace and punctuation, expand a small set of known abbreviations. Catches `"M. Ibarra"` vs `"Maria Ibarra"` only if you already know `M.` expands to `Maria` -- which is exactly the case a hand-written rule can't generalize from. It's included here because it's nearly free and catches real cases (casing, stray whitespace), not because it's sufficient on its own.
2. **Embedding similarity.** Turn each entity name into a vector (an embedding) and merge names whose vectors are close together, within the same node type only -- a `Product` is never merged into a `Segment` even if the text happens to be similar, because comparing only within type is what keeps this step from merging things that only sound alike. This catches `"TrailRunner backpack"` and `"TrailRunner 32L Backpack"`, which share no normalized string but mean the same thing.

Neither pass is perfect. A normalization rule can merge two genuinely different customers who happen to share a nickname; an embedding threshold set too loose merges near-misses that shouldn't merge, set too tight leaves real duplicates apart. Production systems usually add a third pass -- a person reviewing the merge list before it's final -- which is part of what the overview meant by the human cost of keeping a graph's structure honest.

In [ ]:
import re

KNOWN_ALIASES = {
    "m ibarra": "maria ibarra",  # no period -- alias lookup runs after punctuation is stripped below
}


def normalize(name: str) -> str:
    n = name.lower().strip()
    n = re.sub(r"[^\w\s]", "", n)   # drop punctuation
    n = re.sub(r"\s+", " ", n)       # collapse whitespace
    return KNOWN_ALIASES.get(n, n)


# Group by (type, normalized string) first -- this is the free pass.
from collections import defaultdict

normalized_groups = defaultdict(list)
for e in all_entities:
    key = (e["type"], normalize(e["name"]))
    normalized_groups[key].append(e["name"])

print("Entities after normalization pass:", len(normalized_groups), "(from", len(all_entities), "raw)")

In [ ]:
def get_embedding(text: str) -> list:
    """Get an embedding vector for `text`. Only implemented for Gemini in this notebook --
    if you're on a different LLM_PROVIDER, either add that provider's embedding call here
    or skip to the fallback merge in the next cell."""
    from google import genai
    from google.colab import userdata

    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    result = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return result.embeddings[0].values


def cosine_similarity(a: list, b: list) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b)


SIMILARITY_THRESHOLD = 0.90  # tune this: too low merges unrelated entities, too high misses real duplicates

# One representative string per normalized group, plus its embedding.
group_keys = list(normalized_groups.keys())
group_labels = [normalized_groups[k][0] for k in group_keys]  # first-seen spelling as the label
group_embeddings = [get_embedding(f"{k[0]}: {label}") for k, label in zip(group_keys, group_labels)]

print(f"Computed {len(group_embeddings)} embeddings for entity resolution.")

In [ ]:
# Union-find over the normalized groups: merge any two of the same type whose embeddings
# are close enough. canonical_id maps every raw name (already normalized) to one final label.
parent = list(range(len(group_keys)))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(i, j):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[rj] = ri


for i in range(len(group_keys)):
    for j in range(i + 1, len(group_keys)):
        if group_keys[i][0] != group_keys[j][0]:  # only merge within the same node type
            continue
        sim = cosine_similarity(group_embeddings[i], group_embeddings[j])
        if sim >= SIMILARITY_THRESHOLD:
            print(f"Merging ({group_keys[i][0]}) {group_labels[i]!r} + {group_labels[j]!r}  (similarity {sim:.3f})")
            union(i, j)

# canonical_name[type, normalized_string] -> the final display name to use in the graph
canonical_name = {}
for i, key in enumerate(group_keys):
    root = find(i)
    canonical_name[key] = group_labels[root]

print(f"\n{len(set(canonical_name.values()))} canonical entities after both merge passes (from {len(all_entities)} raw extractions).")

In [ ]:
def canonicalize(entity_type: str, raw_name: str) -> str:
    key = (entity_type, normalize(raw_name))
    return canonical_name.get(key, raw_name)


# Apply canonicalization to relationships, and drop any relationship whose endpoints
# didn't survive as valid entities (guards against a malformed extraction).
entity_types_by_norm = {(e["type"], normalize(e["name"])) for e in all_entities}

canonical_relationships = []
seen = set()
for r in all_relationships:
    # We don't know the type of source/target from the relationship dict alone, so look
    # up every entity of matching normalized name regardless of type, and canonicalize each.
    src_matches = [k for k in entity_types_by_norm if k[1] == normalize(r["source"])]
    tgt_matches = [k for k in entity_types_by_norm if k[1] == normalize(r["target"])]
    if not src_matches or not tgt_matches:
        continue
    src_name = canonical_name.get(src_matches[0], r["source"])
    tgt_name = canonical_name.get(tgt_matches[0], r["target"])

    triple = (src_name, r["relation"], tgt_name)
    if triple in seen:  # the same fact often gets extracted from more than one document
        continue
    seen.add(triple)
    canonical_relationships.append({"source": src_name, "relation": r["relation"], "target": tgt_name, "source_doc": r["source_doc"]})

print(f"{len(canonical_relationships)} unique relationships after canonicalization and de-duplication (from {len(all_relationships)} raw).")

## Loading the graph

Cypher's core statement for this is **`MERGE`**, not `CREATE`. `CREATE` always adds a new node or edge, even if an identical one already exists -- run a `CREATE`-based load twice and you get the same graph twice, doubled. `MERGE` checks whether a node or edge matching the given pattern already exists and only creates it if it doesn't, which makes the load **idempotent**: safe to re-run without duplicating anything, useful the first time you fix a bug in the extraction step and need to reload.

Each node gets a **label** (its type, e.g. `Customer`) and a `name` property. Each relationship gets its **type** (e.g. `FILED`) as the edge label itself, not a property -- this is the main way Cypher differs from a table: the relationship *type* is part of the graph's structure, not a column you filter on.

In [ ]:
def load_graph(entities_by_type: dict, relationships: list, driver) -> None:
    with driver.session() as session:
        for entity_type, names in entities_by_type.items():
            for name in names:
                session.run(
                    f"MERGE (n:{entity_type} {{name: $name}})",
                    name=name,
                )
        for r in relationships:
            session.run(
                f"""
                MATCH (a {{name: $source}}), (b {{name: $target}})
                MERGE (a)-[:{r['relation']}]->(b)
                """,
                source=r["source"],
                target=r["target"],
            )


# Group canonical entity names by type for the load.
entities_by_type = defaultdict(set)
for e in all_entities:
    entities_by_type[e["type"]].add(canonicalize(e["type"], e["name"]))

load_graph(entities_by_type, canonical_relationships, driver)
print("Graph loaded.")

In [ ]:
with driver.session() as session:
    node_count = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    by_label = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(*) AS c ORDER BY c DESC"
    ).data()

print(f"{node_count} nodes, {rel_count} relationships.\n")
for row in by_label:
    print(f"  {row['label']:<14} {row['c']}")

## Querying in Cypher

Three queries, increasing in how many hops (relationship traversals) they need. Cypher's basic shape is `MATCH (pattern) WHERE (filter) RETURN (result)` -- `MATCH` describes a shape to look for in the graph using ASCII-art-like syntax, `()` for a node and `-[]->` for a directed relationship, and Neo4j finds every place in the graph that shape occurs.

**One hop** -- every ticket about a specific product:

In [ ]:
with driver.session() as session:
    result = session.run(
        """
        MATCH (t:Ticket)-[:ABOUT_PRODUCT]->(p:Product {name: $product})
        RETURN t.name AS ticket
        """,
        product="TrailRunner 32L Backpack",
    )
    for row in result:
        print(row["ticket"])

**Two hops** -- every customer who filed a ticket about that product (Customer→Ticket→Product):

In [ ]:
with driver.session() as session:
    result = session.run(
        """
        MATCH (c:Customer)-[:FILED]->(t:Ticket)-[:ABOUT_PRODUCT]->(p:Product {name: $product})
        RETURN DISTINCT c.name AS customer
        """,
        product="TrailRunner 32L Backpack",
    )
    for row in result:
        print(row["customer"])

**Multi-hop, cross-document** -- the actual question a support lead needs answered: which customers complained about the TrailRunner backpack, what kind of issue did each report, and what does the policy actually entitle them to. This single query walks Customer→Ticket→Product, Ticket→IssueType, and PolicyClause→IssueType -- three relationship types sourced from three different original documents (a support ticket, the policy doc, and the product page that named the product in the first place):

In [ ]:
with driver.session() as session:
    result = session.run(
        """
        MATCH (c:Customer)-[:FILED]->(t:Ticket)-[:ABOUT_PRODUCT]->(p:Product {name: $product})
        MATCH (t)-[:HAS_ISSUE_TYPE]->(issue:IssueType)
        OPTIONAL MATCH (clause:PolicyClause)-[:GOVERNS]->(issue)
        RETURN c.name AS customer, t.name AS ticket, issue.name AS issue_type, clause.name AS policy_clause
        """,
        product="TrailRunner 32L Backpack",
    )
    for row in result:
        print(f"{row['customer']:<16} {row['ticket']:<14} {row['issue_type']:<22} -> {row['policy_clause']}")

If that returned two different policy clauses for two different customers on the *same product*, that's the point: a defect and a shipping delay are governed by different clauses with different remedies, and no single document states that mapping for these specific customers. The graph produced it by combining a ticket, a product reference, and a policy document that had never been placed next to each other before.

## What a graph answers that a table struggles with, and where a table still wins

The query above is the case for a graph: the question needed a variable, unknown-in-advance number of hops (customer → ticket → product, then ticket → issue type → policy clause) across data that started in four unrelated documents. A relational table answers a fixed-shape question fast because someone decided the shape (the columns) in advance. A graph answers a question shaped like *"follow this relationship, then whatever it's connected to, then whatever that's connected to"* without anyone having pre-decided how many hops the question would need -- which is exactly what "which customers... under what policy... during which campaign" needed here, and would need a new table, or a new join, or a new column, every time the question's shape changed even slightly.

That advantage doesn't hold in the other direction. **"What's the average order value for the Weekend Hikers segment this quarter"** is an aggregation over many rows of the same shape -- a `GROUP BY` in SQL, or an equivalent Cypher aggregation -- and a warehouse table built for that column will answer it faster and more cheaply than a graph traversal will, because the graph has to walk relationships to reach data a table would have sitting in a single indexed column. The honest framing from this week's overview holds: decide which structure fits by the shape of the query you actually need answered, not by which technology is newer. A team that puts every table into a graph "for flexibility" pays a real cost in query speed and infrastructure for questions a table would have answered directly.

## GraphRAG: using the graph as a model's source of facts

**RAG** (retrieval-augmented generation) is the general pattern: before asking a model to answer, retrieve the specific information the answer needs and hand it to the model as context, rather than trusting whatever the model already "knows." The retrieval step is usually a similarity search over document chunks. **GraphRAG** swaps that retrieval step for a graph traversal: instead of handing the model a paragraph that might contain the answer, hand it the actual subgraph -- the specific nodes and relationships -- the question depends on.

The difference matters most exactly where the earlier section did: a question that needs several connected facts assembled from different places. A chunk-similarity search retrieves whichever paragraphs read as topically similar to the question, which is not the same as retrieving the specific chain of relationships the answer depends on -- similarity is about *what a passage is about*, not about *which entities connect to which*. A graph traversal retrieves the second thing directly.

This implementation asks the model to write the Cypher query itself, given the schema -- a real, named technique usually called **text-to-Cypher**. That's powerful and worth treating carefully: an LLM-written query that runs unchecked against a live database is a real risk if the query could modify data, which is why the function below refuses to run anything except a read (no `CREATE`, `MERGE`, `DELETE`, `SET`, or `REMOVE`) before it ever reaches the database. This is the same instinct Session 3 covered under a different name -- validate anything an LLM produces before it gets to act, rather than trusting the output because it's syntactically well-formed.

In [ ]:
SCHEMA_DESCRIPTION = """
Node types: Customer, Product, Ticket, Campaign, Segment, IssueType, PolicyClause, Company
Relationship types (source_type -RELATION-> target_type):
  Customer -FILED-> Ticket
  Ticket -ABOUT_PRODUCT-> Product
  Ticket -HAS_ISSUE_TYPE-> IssueType
  PolicyClause -GOVERNS-> IssueType
  Customer -IN_SEGMENT-> Segment
  Campaign -TARGETS_SEGMENT-> Segment
  Campaign -FEATURES_PRODUCT-> Product
  Ticket -DURING_CAMPAIGN-> Campaign
  Product -MANUFACTURED_BY-> Company
All nodes have a `name` property.
"""

WRITE_KEYWORDS = ("CREATE", "MERGE", "DELETE", "SET", "REMOVE", "DROP", "DETACH")


def question_to_cypher(question: str) -> str:
    prompt = f"""Given this graph schema:
{SCHEMA_DESCRIPTION}

Write ONE Cypher query that retrieves the nodes and relationships needed to answer this
question: "{question}"

Return every node's `name` property and the relationship types connecting them, so the
result can be read as a set of facts. Return ONLY the Cypher query, no explanation, no
markdown fences."""
    cypher = call_llm(prompt).strip().strip("`")
    if cypher.lower().startswith("cypher"):
        cypher = cypher[6:].strip()
    return cypher


def run_read_only(cypher: str, driver) -> list:
    upper = cypher.upper()
    if any(kw in upper for kw in WRITE_KEYWORDS):
        raise ValueError(f"Refusing to run a query containing a write keyword:\n{cypher}")
    with driver.session() as session:
        return session.run(cypher).data()

In [ ]:
def graph_rag_answer(question: str, driver) -> dict:
    """Retrieve a subgraph relevant to `question` via text-to-Cypher, then answer using
    only that subgraph as context. Returns the answer alongside the raw retrieval, so the
    next section can check whether the answer is actually grounded in it."""
    cypher = question_to_cypher(question)
    try:
        rows = run_read_only(cypher, driver)
    except Exception as e:
        return {"question": question, "cypher": cypher, "rows": [], "answer": f"Retrieval failed: {e}"}

    context = "\n".join(str(row) for row in rows) if rows else "(no matching data found in the graph)"

    answer_prompt = f"""Answer the question using ONLY the facts below. If the facts don't
fully answer it, say plainly what's missing rather than filling the gap from general
knowledge.

Facts retrieved from the knowledge graph:
{context}

Question: {question}

Answer:"""
    answer = call_llm(answer_prompt)
    return {"question": question, "cypher": cypher, "rows": rows, "answer": answer}

In [ ]:
questions = [
    "Which customers reported a problem with the TrailRunner 32L Backpack, and what kind of issue did each report?",
    "Was Priya Natarajan's order part of the Summer Trail Sale campaign?",
]

results = []
for q in questions:
    r = graph_rag_answer(q, driver)
    results.append(r)
    print("Q:", r["question"])
    print("Cypher used:", r["cypher"].strip().replace("\n", " "))
    print("Rows retrieved:", len(r["rows"]))
    print("A:", r["answer"])
    print("-" * 70)

## Checking groundedness

An answer that reads confidently is not the same as an answer that's actually supported by what was retrieved. The point of GraphRAG is that the model should only be asserting what the subgraph contains -- so a useful check, even a rough one, is whether the entity names in the answer actually appear among the entity names retrieved. This won't catch every kind of error (a model can misstate a *relationship* between two entities that were both genuinely retrieved), but it catches the most damaging failure: an answer that names a customer, product, or policy clause that was never in the retrieved subgraph at all, which means the model filled a gap from training data or invention rather than from the graph.

In [ ]:
def groundedness_check(result: dict) -> dict:
    retrieved_names = set()
    for row in result["rows"]:
        for value in row.values():
            if isinstance(value, str):
                retrieved_names.add(value.lower())

    # Very rough: flag any retrieved name that the answer does NOT mention, so a person can
    # see what was available but unused, and separately confirm the answer isn't citing
    # something absent from retrieved_names entirely. This is a starting checklist, not a
    # proof -- a person should still read the answer against the facts before trusting it.
    answer_lower = result["answer"].lower()
    mentioned = {name for name in retrieved_names if name in answer_lower}
    unused = retrieved_names - mentioned

    return {
        "retrieved_entity_count": len(retrieved_names),
        "entities_the_answer_used": sorted(mentioned),
        "retrieved_but_unused": sorted(unused),
    }


for r in results:
    check = groundedness_check(r)
    print("Q:", r["question"])
    print(" retrieved entities:", check["retrieved_entity_count"])
    print(" answer used:", check["entities_the_answer_used"])
    print(" retrieved but not mentioned in the answer:", check["retrieved_but_unused"])
    print("-" * 70)

"Retrieved but not mentioned" is not automatically a problem -- the model may have correctly judged some retrieved facts irrelevant to the specific question. What would be a problem, and what this check is actually built to catch, is the reverse: a name in the answer that never shows up in `retrieved_names` at all. Run the cell again with a question the graph genuinely can't answer (try `"What is Fernwood's revenue this quarter?"`, which has no node in this schema) and check whether the model says so plainly instead of guessing -- that's the behavior the answer prompt above was written to push toward, and it's worth confirming it actually holds rather than assuming the prompt worked.

## Does this actually improve on the alternatives?

Groundedness checking works without knowing the real answer in advance -- it only asks whether the model stayed inside what it was given. This section asks a different, stronger question, one only possible here because Fernwood Outfitters was invented for this exercise and the correct answer is therefore fully known in advance: **does GraphRAG produce a more correct answer than the alternatives actually would have?**

Three approaches, same question, same underlying facts:

1. **No retrieval at all** -- ask the model directly, nothing else.
2. **Vector similarity RAG** -- the standard pattern: embed each document, embed the question, retrieve the top-k most similar documents, hand them to the model as context.
3. **GraphRAG** -- the graph traversal built above.

The question needs facts from two different tickets and the policy document at once: *"for each customer who reported a problem with the TrailRunner 32L Backpack, what kind of issue did they report, and what does Fernwood's policy entitle them to as a result?"* The correct answer, from the source documents directly: Maria Ibarra reported a manufacturing defect; David Chen reported a shipping delay. Those two issue types are what the policy actually keys its remedy off of, so getting a customer's issue type right is the fact that determines whether the rest of the answer can be right at all. It's also the specific, checkable error this section watches for in a way remedy wording alone can't: Fernwood's policy phrases both remedies using the words "full refund" and "free return," so a check based on remedy text can't tell a correct answer from a swapped one, but "manufacturing defect" and "shipping delay" share no words -- attributing the wrong one to a customer is unambiguous and worse than an incomplete answer, because it states something specific and wrong rather than simply leaving a gap.

In [ ]:
PAYOFF_QUESTION = (
    "For each customer who reported a problem with the TrailRunner 32L Backpack, "
    "what kind of issue did they report, and what does Fernwood's policy entitle "
    "them to as a result?"
)

# Known correct answer, established directly from the source documents (support tickets
# #1042 and #1077, and the policy document) -- not from anything the model produced.
GROUND_TRUTH = {
    "maria ibarra": {"issue_type": "manufacturing defect"},
    "david chen": {"issue_type": "shipping delay"},
}
# Note on what this checks and why: the policy's two remedies actually share language --
# "full refund" and "free return" both appear in BOTH clauses (the shipping-delay remedy
# is "a 20% refund... or a free return with full refund"), so checking for remedy wording
# can't reliably tell the two apart. The issue TYPE can: "manufacturing defect" and
# "shipping delay" share no words, and the issue type is what determines the remedy in the
# first place, so getting it right per customer is the fact that actually matters here.


def score_answer(answer: str) -> dict:
    """Check whether each customer's sentence(s) in `answer` state THEIR OWN correct issue
    type, the OTHER customer's issue type (a swap -- worse than a gap, since it states
    something specific and wrong), both (ambiguous), or neither (incomplete). This is a
    keyword check, not real language understanding -- like `groundedness_check` above, it
    catches a real class of error but isn't proof an answer is correct in every other way."""
    sentences = re.split(r"(?<=[.!?])\s+", answer)
    results = {}
    for customer, expected in GROUND_TRUTH.items():
        customer_text = " ".join(s for s in sentences if customer in s.lower()).lower()
        if not customer_text:
            results[customer] = "not mentioned"
            continue
        other_issue_type = next(c["issue_type"] for name, c in GROUND_TRUTH.items() if name != customer)
        has_correct = expected["issue_type"] in customer_text
        has_wrong = other_issue_type in customer_text
        if has_wrong and not has_correct:
            results[customer] = "WRONG (attributed the other customer's issue type)"
        elif has_correct and has_wrong:
            results[customer] = "ambiguous (states both issue types)"
        elif has_correct:
            results[customer] = "correct issue type"
        else:
            results[customer] = "incomplete (issue type not stated)"
    return results

### Approach 1: no retrieval

Fernwood Outfitters, both customers, and both tickets were invented for this course and exist nowhere else -- not in any model's training data, not on the web. A model asked this question with no context has, honestly, nothing to draw on. The only fully correct response is a plain statement that it doesn't have this information; anything more specific than that is fabricated by construction, regardless of how confident it reads.

In [ ]:
naive_answer = call_llm(PAYOFF_QUESTION)
print(naive_answer)
print("\nScore:", score_answer(naive_answer))

### Approach 2: vector similarity RAG

Embed every document once, embed the question, rank documents by cosine similarity, and hand the model the top few as context -- the retrieval pattern behind most RAG systems, reusing the same `get_embedding()` used earlier for entity resolution. `TOP_K` is set to 2 here on purpose: a real system can't hand a model every document in a large corpus, so it fixes a cutoff, and this question actually needs three documents (two tickets plus the policy) to answer completely. Watch which two make the cutoff, and whether the model still tries to answer as if it had everything.

In [ ]:
doc_embeddings = {d["doc_id"]: get_embedding(d["text"]) for d in DOCS}
question_embedding = get_embedding(PAYOFF_QUESTION)

ranked = sorted(
    DOCS,
    key=lambda d: cosine_similarity(question_embedding, doc_embeddings[d["doc_id"]]),
    reverse=True,
)

TOP_K = 2
retrieved_docs = ranked[:TOP_K]

print("Ranked by similarity to the question:")
for d in ranked:
    sim = cosine_similarity(question_embedding, doc_embeddings[d["doc_id"]])
    flag = "  <- retrieved" if d in retrieved_docs else ""
    print(f"  {sim:.3f}  {d['doc_id']}{flag}")

In [ ]:
vector_rag_context = "\n\n".join(d["text"] for d in retrieved_docs)

vector_rag_prompt = f"""Answer the question using ONLY the documents below. If they don't
fully answer it, say plainly what's missing rather than filling the gap from general
knowledge.

Documents:
{vector_rag_context}

Question: {PAYOFF_QUESTION}

Answer:"""

vector_rag_answer = call_llm(vector_rag_prompt)
print(vector_rag_answer)
print("\nScore:", score_answer(vector_rag_answer))

If a needed document didn't make the `TOP_K` cutoff above, that's not a bug in the retrieval code -- it's the structural limit of ranking by similarity to the question as a whole. The policy document's text is about shipping SLAs and manufacturing defects in the abstract; it never mentions "TrailRunner" by name, so nothing in its embedding ties it to *this specific product's* tickets except a fact a similarity score can't see: an explicit `GOVERNS` edge from a policy clause to an issue type, which only exists in the graph. And if it did make the cutoff, that alone still doesn't guarantee a correct answer -- the model still has to work out, from a pile of undifferentiated raw text, which clause governs which customer's issue without crossing the two. Vector RAG's retrieval step and its reasoning step are both exposed to error here in a way the graph traversal isn't, because the graph did the "which fact connects to which" step as an explicit, deterministic query before the model ever saw text.

### Approach 3: GraphRAG

Same question, run through `graph_rag_answer()` from the section above -- the graph traversal assembles the exact subgraph the question needs regardless of which words the source documents happen to share with each other, because it follows typed edges rather than text similarity.

In [ ]:
graphrag_result = graph_rag_answer(PAYOFF_QUESTION, driver)
print(graphrag_result["answer"])
print("\nScore:", score_answer(graphrag_result["answer"]))

### Comparing all three

In [ ]:
comparison = {
    "No retrieval": score_answer(naive_answer),
    f"Vector RAG (top {TOP_K})": score_answer(vector_rag_answer),
    "GraphRAG": score_answer(graphrag_result["answer"]),
}

print(f"{'Approach':<20}{'Maria Ibarra':<55}{'David Chen'}")
print("-" * 115)
for approach, scores in comparison.items():
    print(f"{approach:<20}{scores['maria ibarra']:<55}{scores['david chen']}")

Read the table your own run actually produced, not a result asserted here in advance -- the point of building all three is to measure, not to assume. What the setup guarantees regardless of the specific wording each run produces: the no-retrieval approach has no way to be right beyond a lucky guess, since the facts exist nowhere for it to draw on; the vector RAG approach depends entirely on whether similarity ranking happened to surface the connecting document within whatever cutoff was set, and, even then, on the model correctly keeping two customers' facts apart while reading a pile of undifferentiated text; and the GraphRAG approach retrieved the exact subgraph needed because the schema already encodes which policy clause governs which issue type, independent of word choice in the source text. That last point is the actual mechanism behind "a graph improves factual accuracy" -- it isn't that a graph makes a model smarter, it's that a graph turns a fact-assembly step the model would otherwise have to infer from raw text into a deterministic database lookup performed before the model is ever asked to reason at all.

One question and one graph is a demonstration, not a benchmark -- treat this as showing the mechanism, not as a claim that GraphRAG wins by some fixed margin in general. The honest generalization is narrower and still real: any question whose answer depends on connecting facts across documents that don't share much wording is a question where similarity-based retrieval is structurally exposed to missing the connection, and a graph traversal isn't, because it was never relying on wording in the first place.

## What this costs

**In model calls**, this notebook alone made roughly: 10 extraction calls (one per document), one embedding call per canonical entity group during merging, one Cypher-generation call plus one answer call per GraphRAG question. None of that is expensive on a lightweight model -- Session 1's token cost guide covers the exact method (`count_tokens` plus a per-1M-token price) and its own note that pricing changes, so check [AI Studio](https://aistudio.google.com) or your provider's current pricing page rather than trusting a number written on a fixed date. What scales here is call *count*, not call *size* -- a real internal-documentation set of a few thousand documents means a few thousand extraction calls, which is where the estimate actually needs doing before a real run, not after.

**In human work**, which is the cost the overview flagged as easy to undercount: someone has to define the schema before extraction starts (done once, here, but revisited every time a new document type shows up), review the merge decisions entity resolution makes (the `SIMILARITY_THRESHOLD` constant above is a judgment call, not a fact), and re-run extraction whenever a source document changes, since nothing here keeps the graph in sync with its sources automatically. A graph that was accurate the day it was built and never touched again degrades exactly the way a spreadsheet does when nobody keeps updating it -- except a stale graph looks structurally complete right up until someone traverses the specific edge that's now wrong.

In [ ]:
# Rough token-count check on the extraction step, using the pattern from Session 1's
# token cost guide -- run this before pointing extraction at a real, much larger document set.
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

sample_prompt = EXTRACTION_PROMPT.format(doc_type=DOCS[0]["doc_type"], doc_id=DOCS[0]["doc_id"], text=DOCS[0]["text"])
token_count = client.models.count_tokens(model="gemini-2.5-flash-lite", contents=sample_prompt)
print(f"One extraction call on a document this size: ~{token_count.total_tokens} input tokens.")
print(f"For N documents of similar length: roughly N x {token_count.total_tokens} input tokens, before output.")
print("Check current per-token pricing at https://aistudio.google.com before scaling this up.")

## What you built

A graph assembled from ten documents that were never designed to reference each other, queried across three hops in one Cypher statement, and used as the grounding source for a model's answer instead of the model's own memory -- with a way to check whether the answer actually stayed grounded, and a side-by-side comparison against no retrieval and against standard vector RAG showing the specific mechanism behind why the graph holds up better on a question like this one: the fact-assembly step happened as a deterministic query before the model ever reasoned over text, rather than being left for the model to infer from a pile of retrieved paragraphs.

The gaps worth carrying forward, not glossing over: entity resolution here used one similarity threshold tuned by eye on ten documents, which will not hold unchanged at real-document-set scale; the text-to-Cypher step trusts the model to write a correct query and only guards against a *destructive* one, not a wrong-but-safe one that retrieves the wrong subgraph without any error to flag it; and the groundedness check catches missing entities, not misstated relationships between entities that were genuinely retrieved. Each of those is a real, open problem in production GraphRAG systems, not a simplification unique to this notebook -- worth knowing going in, rather than discovering the first time a stakeholder asks a question the graph answers confidently and wrongly.